# LLM-Based Assessment System

# Imports

In [ ]:
import json
import asyncio
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Dict, List, Any

# Response Schema

In [2]:
class DimensionScore(BaseModel):
    score: int
    reasoning: str

class TaskAssessment(BaseModel):
    task_predictability: DimensionScore
    interaction_medium: DimensionScore
    social_requirement: DimensionScore
    environmental_stability: DimensionScore
    consequence_of_failure: DimensionScore
    regulatory_barrier: DimensionScore
    economic_arbitrage: DimensionScore

# Gemini Automation Pipeline Class

In [ ]:
class AutomationAssessmentPipeline:
    def __init__(self, api_key: str, framework_path: str, data_path: str, max_concurrent: int = 2):
        # Initialize official Google GenAI Async client
        self.client = genai.Client(api_key=api_key)
        self.framework_path = framework_path
        self.data_path = data_path
        
        # Free tier semaphore to prevent HTTP 429 (Rate Limit Exceeded) errors
        self.semaphore = asyncio.Semaphore(max_concurrent)
        
        self.framework_dimensions = self._load_framework()
        self.job_data = self._load_json_data()

    def _load_framework(self) -> str:
        """Reads the framework Excel file."""
        df = pd.read_excel(self.framework_path)
        return df.to_string(index=False)

    def _load_json_data(self) -> List[Dict[str, Any]]:
        """Reads the job JSON dataset."""
        with open(self.data_path, 'r') as file:
            return json.load(file)

    def build_assessment_prompt(self, task_description: str, job_context: str) -> str:
        """Constructs prompt using context and scoring criteria."""
        return f"""
        Analyze the following job task and evaluate its automation potential across the provided dimensions.
        
        Framework Dimensions and Scoring Criteria:
        {self.framework_dimensions}
        
        Job Context: {job_context}
        Task: {task_description}
        
        Be specific and justify each score strictly based on the framework criteria.
        """

    async def assess_task_automation(self, task_description: str, job_context: str) -> Dict[str, Any]:
        """Calls Gemini API asynchronously with structured JSON enforcement."""
        prompt = self.build_assessment_prompt(task_description, job_context)
        
        async with self.semaphore:
            try:
                # Small pause to maintain free tier RPM compliance (15 RPM max)
                await asyncio.sleep(4.0)
                
                response = await self.client.aio.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                        response_mime_type="application/json",
                        response_schema=TaskAssessment,
                        system_instruction="You are an expert in labor economics and AI automation."
                    )
                )
                return json.loads(response.text)
            except Exception as e:
                print(f"Error processing task '{task_description[:30]}...': {e}")
                return {}

    def calculate_automation_score(self, dimension_scores: Dict[str, Any], weights: Dict[str, float]) -> float:
        """Calculates normalized weighted automation potential (0-100%)."""
        overall_score = 0.0
        for dim, details in dimension_scores.items():
            if dim in weights and isinstance(details, dict) and 'score' in details:
                normalized_score = (details['score'] / 5.0) * 100
                overall_score += normalized_score * weights[dim]
        return overall_score

    async def process_all_occupations(self, weights: Dict[str, float]) -> List[Dict[str, Any]]:
        """Main loop managing concurrent evaluations across tasks."""
        results = []
        for job in self.job_data:
            job_title = job.get('job_title', 'Unknown')
            job_desc = job.get('job_description', '')
            job_context = f"{job_title} - {job_desc}"
            
            coroutines = [
                self.assess_task_automation(task['task_description'], job_context) 
                for task in job.get('tasks', [])
            ]
            
            task_assessments = await asyncio.gather(*coroutines)
            
            for task, assessment in zip(job.get('tasks', []), task_assessments):
                if assessment:
                    overall_score = self.calculate_automation_score(assessment, weights)
                    results.append({
                        "job_title": job_title,
                        "task_id": task['task_id'],
                        "task_description": task['task_description'],
                        "dimension_assessments": assessment,
                        "overall_automation_score": overall_score
                    })
        return results